# 사업보고서 청크 임베딩 — BGE-M3 + ChromaDB

**입력**: `VAR/KOSPI/*.jsonl` + `VAR/KOSDAQ/*.jsonl`
**출력**: `VAR/embedding/chroma_db/`
**모델**: BGE-M3 via transformers.AutoModel (FP16 on GPU)

## 파이프라인
1. 환경 확인
2. 청크 로드 (KOSPI + KOSDAQ)
3. BGE-M3 모델 로드 (FP16)
4. 증분 필터링
5. 배치 임베딩 + ChromaDB 저장 (batch 64)
6. 완료 통계
7. 검증 검색

**중간에 멈춰도 안전**: 증분 업데이트 ON. 다시 실행하면 이어서 진행.

## 1. 환경 확인 + 경로 설정

In [1]:
import os, sys
from pathlib import Path

try:
    sys.stdout.reconfigure(line_buffering=True)
except Exception:
    pass

VAR_ROOT = Path(r'C:\Users\Admin\Desktop\VAR')
os.chdir(VAR_ROOT)
if str(VAR_ROOT) not in sys.path:
    sys.path.insert(0, str(VAR_ROOT))

print(f'cwd      : {os.getcwd()}')
print(f'Python   : {sys.version.split()[0]}')

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}', end='')
if torch.cuda.is_available():
    print(f'  ({torch.cuda.get_device_name(0)}, '
          f'VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB)')
else:
    print('  ⚠ GPU 미사용')

cwd      : C:\Users\Admin\Desktop\VAR
Python   : 3.11.15
PyTorch  : 2.6.0+cu124
CUDA     : True  (NVIDIA GeForce RTX 4060 Laptop GPU, VRAM 8.6GB)


## 2. 청크 로드 (KOSPI + KOSDAQ)

예상: 약 192만 청크

In [2]:
from embedding.chunk_loader import load_all_chunks

chunks, stats = load_all_chunks(verbose=True)
print(f'\n로드 완료: {len(chunks):,} 청크')

[chunk_loader] 2596개 파일 발견
  [50/2596] 로드 중... 누적 30,288개 청크
  [100/2596] 로드 중... 누적 59,342개 청크
  [150/2596] 로드 중... 누적 90,553개 청크
  [200/2596] 로드 중... 누적 127,794개 청크
  [250/2596] 로드 중... 누적 163,373개 청크
  [300/2596] 로드 중... 누적 198,967개 청크
  [350/2596] 로드 중... 누적 234,054개 청크
  [400/2596] 로드 중... 누적 266,669개 청크
  [450/2596] 로드 중... 누적 302,020개 청크
  [500/2596] 로드 중... 누적 335,679개 청크
  [550/2596] 로드 중... 누적 367,984개 청크
  [600/2596] 로드 중... 누적 403,065개 청크
  [650/2596] 로드 중... 누적 438,832개 청크
  [700/2596] 로드 중... 누적 477,153개 청크
  [750/2596] 로드 중... 누적 510,660개 청크
  [800/2596] 로드 중... 누적 544,894개 청크
  [850/2596] 로드 중... 누적 577,038개 청크
  [900/2596] 로드 중... 누적 612,462개 청크
  [950/2596] 로드 중... 누적 644,961개 청크
  [1000/2596] 로드 중... 누적 677,612개 청크
  [1050/2596] 로드 중... 누적 712,003개 청크
  [1100/2596] 로드 중... 누적 745,187개 청크
  [1150/2596] 로드 중... 누적 777,636개 청크
  [1200/2596] 로드 중... 누적 810,673개 청크
  [1250/2596] 로드 중... 누적 843,325개 청크
  [1300/2596] 로드 중... 누적 875,586개 청크
  [1350/2596] 로드 중... 누적 906,137개 

## 3. BGE-M3 모델 로드 (FP16)

In [3]:
# 모델 캐시 초기화 (FP32 로 미리 로드된 게 있으면 FP16 재로드 보장)
from embedding import embedder
embedder._MODEL = None
embedder._TOKENIZER = None

from embedding.embedder import get_device_info, _load_model

info = get_device_info()
print(f'Device : {info["device"]}')
print(f'Model  : {info["model"]}')
print(f'Dim    : {info["dim"]}')
if info['device'] == 'cuda':
    print(f'GPU    : {info.get("gpu_name","?")}  '
          f'(VRAM {info.get("gpu_mem_total_gb","?"):.1f}GB)')

print('\n모델 로딩 중...')
_load_model()
print('✓ 모델 로드 완료 (FP16)')

Device : cuda
Model  : BAAI/bge-m3
Dim    : 1024
GPU    : NVIDIA GeForce RTX 4060 Laptop GPU  (VRAM 8.6GB)

모델 로딩 중...
[embedder] loading model: BAAI/bge-m3 (device=cuda)
[embedder] loaded (dim=1024, precision=FP16)
✓ 모델 로드 완료 (FP16)


## 4. 증분 필터링

ChromaDB 에 이미 임베딩된 ID 는 자동 스킵.  
(이전 21 청크/sec 로 돌린 부분 자동 인계)

In [4]:
from embedding.vector_store import get_existing_ids, get_stats

all_ids = [c['id'] for c in chunks]
existing_ids = set()
CHUNK = 5000
for i in range(0, len(all_ids), CHUNK):
    existing_ids.update(get_existing_ids(all_ids[i:i+CHUNK]))

new_chunks = [c for c in chunks if c['id'] not in existing_ids]
print(f'전체 청크   : {len(chunks):>9,}')
print(f'이미 임베딩 : {len(existing_ids):>9,}')
print(f'신규 대상   : {len(new_chunks):>9,}')
print(f'\nDB 현황: {get_stats()}')

전체 청크   : 1,922,942
이미 임베딩 :     5,120
신규 대상   : 1,917,822

DB 현황: {'collection': 'annual_reports', 'total_chunks': 5120, 'db_path': 'C:\\Users\\Admin\\Desktop\\VAR\\embedding\\chroma_db'}


## 5. 배치 임베딩 + ChromaDB 저장

- 임베딩 배치: **64** (FP16 으로 VRAM 절반, batch 더 키울 수 있음)
- DB 쓰기 배치: 256
- 예상 속도: 200~400 청크/sec (이전 21 청크/sec 의 10~20배)
- ETA: 약 1.5~3시간 (192만 신규 기준)

OOM 시 EMBED_BATCH 를 32 또는 16 으로 줄임.

In [5]:
from tqdm.auto import tqdm
from embedding.embedder import embed_texts
from embedding.vector_store import add_batch

EMBED_BATCH = 64       # OOM 시 32 또는 16
WRITE_BATCH = EMBED_BATCH * 4   # 256

total = len(new_chunks)
pbar = tqdm(total=total, desc='Embedding', unit='chunks')

for i in range(0, total, WRITE_BATCH):
    batch_chunks = new_chunks[i:i+WRITE_BATCH]
    texts     = [c['text'] for c in batch_chunks]
    ids       = [c['id'] for c in batch_chunks]
    metadatas = [c['metadata'] for c in batch_chunks]

    embeddings = embed_texts(texts, batch_size=EMBED_BATCH, show_progress=False)
    embeddings_list = embeddings.tolist()

    add_batch(ids, texts, embeddings_list, metadatas)

    pbar.update(len(batch_chunks))

pbar.close()
print('\n✓ 임베딩 완료')

Embedding:   0%|          | 0/1917822 [00:00<?, ?chunks/s]


✓ 임베딩 완료


## 6. 완료 통계

In [6]:
stats_db = get_stats()
print(f'컬렉션      : {stats_db["collection"]}')
print(f'총 청크 수  : {stats_db["total_chunks"]:,}')
print(f'DB 경로     : {stats_db["db_path"]}')

컬렉션      : annual_reports
총 청크 수  : 1,922,942
DB 경로     : C:\Users\Admin\Desktop\VAR\embedding\chroma_db


## 7. 검증 검색

In [ ]:
from embedding.retrieval import retrieve

print('=' * 60)
print('Q: 현대건설 — 주요 사업 부문 및 매출 구성')
print('=' * 60)
results = retrieve(
    query='주요 사업 부문 매출 구성 도급 주택 플랜트',
    ticker='000720', year=2025, top_k=3,
)
for i, r in enumerate(results, 1):
    m = r['metadata']
    print(f'\n[{i}] sim={r["similarity"]:.3f}  '
          f'{m.get("corp_name","")} | {m.get("section_path_str","")}')
    print(f'    {r["text"][:200]}...')

Q: 현대건설 — 주요 사업 부문 및 매출 구성


In [ ]:
print('=' * 60)
print('Q: 삼양식품 — 신용평가 등급')
print('=' * 60)
results = retrieve(
    query='신용평가 등급 회사채 NICE 한국기업평가',
    ticker='003230', year=2025, top_k=3,
)
for i, r in enumerate(results, 1):
    m = r['metadata']
    print(f'\n[{i}] sim={r["similarity"]:.3f}  '
          f'{m.get("corp_name","")} | {m.get("section_path_str","")}')
    print(f'    {r["text"][:200]}...')